In [1]:
! pip install optuna


   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   -------- ------------------------------- 1/5 [Mako]
   -------- ------------------------------- 1/5 [Mako]
   -------- ------------------------------- 1/5 [Mako]
   -------- ------------------------------- 1/5 [Mako]
   -------- ------------------------------- 1/5 [Mako]
   -------- ------------------------------- 1/5 [Mako]
   ------------------------ --------------- 3/5 [alembic]
   ------------------------ --------------- 3/5 [alembic]
   ------------------------ --------------- 3/5 [alembic]
   ------------------------ --------------- 3/5 [alembic]
   ------------------------ --------------- 3/5 [alembic]
   ------------------------ --------------- 3/5 [alembic]
   ------------------------ --------------- 3/5 [alembic]
   ------------------------ ---------------

In [9]:
# ============================================================
# NOTEBOOK 2 — Hyperparameter Tuning
# Models : RandomForest, SVM
# Methods : GridSearchCV, RandomizedSearchCV, Optuna
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.datasets import load_breast_cancer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

#import optuna
#from optuna.integration import SklearnPipelineSampler

import warnings
warnings.filterwarnings("ignore")

data = load_breast_cancer()
X = data.data
y = data.target


* 1. Baseline model

In [ ]:
pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC())
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

pipe_svm.fit(X_train, y_train)
pipe_svm.score(X_test, y_test)


0.9790209790209791

2. GridSearchCV sur SVM

In [11]:
grid_params = {
    "model__C": [0.1, 1, 10],
    "model__gamma": ["scale", "auto"],
    "model__kernel": ["rbf", "linear"]
}

grid = GridSearchCV(pipe_svm, grid_params, cv=5, n_jobs=-1)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Test accuracy:", grid.score(X_test, y_test))


Best parameters: {'model__C': 0.1, 'model__gamma': 'scale', 'model__kernel': 'linear'}
Test accuracy: 0.986013986013986


3. RandomizedSearchCV

In [12]:
# rand_params = {
#     "model__C": np.logspace(-3, 3, 20),
#     "model__gamma": np.logspace(-4, 1, 20),
#     "model__kernel": ["rbf"]
# }

rand_params = {
    "model__C": [0.1, 1, 10],
    "model__gamma": ["scale", "auto"],
    "model__kernel": ["rbf", "linear"]
}

rand = RandomizedSearchCV(
    pipe_svm, rand_params, n_iter=25, cv=5, n_jobs=-1, random_state=42)
rand.fit(X_train, y_train)

print("Best parameters:", rand.best_params_)
print("Test accuracy:", rand.score(X_test, y_test))

Best parameters: {'model__kernel': 'linear', 'model__gamma': 'scale', 'model__C': 0.1}
Test accuracy: 0.986013986013986


4. Tuning avec OPTUNA (optimisation bayésienne)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)


In [14]:
import optuna
#from optuna.integration import SklearnPipelineSampler
def objective(trial):
    C = trial.suggest_loguniform("model__C", 1e-3, 1e3)
    gamma = trial.suggest_loguniform("model__gamma", 1e-4, 1)
    kernel = trial.suggest_categorical("model__kernel", ["rbf", "linear"])

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(C=C, gamma=gamma, kernel=kernel))
    ])

    pipe.fit(X_train, y_train)
    return pipe.score(X_test, y_test)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

study.best_params


[I 2026-01-22 16:20:20,834] A new study created in memory with name: no-name-01c0b650-0402-4f88-b03a-f48be4dea701
[I 2026-01-22 16:20:20,848] Trial 0 finished with value: 0.9440559440559441 and parameters: {'model__C': 28.466845650535777, 'model__gamma': 0.09867025523802173, 'model__kernel': 'linear'}. Best is trial 0 with value: 0.9440559440559441.
[I 2026-01-22 16:20:20,860] Trial 1 finished with value: 0.6293706293706294 and parameters: {'model__C': 0.0011822256452305921, 'model__gamma': 0.12071522413184939, 'model__kernel': 'rbf'}. Best is trial 0 with value: 0.9440559440559441.
[I 2026-01-22 16:20:20,866] Trial 2 finished with value: 0.9440559440559441 and parameters: {'model__C': 0.0018629085427786173, 'model__gamma': 0.0032961074865323872, 'model__kernel': 'linear'}. Best is trial 0 with value: 0.9440559440559441.
[I 2026-01-22 16:20:20,876] Trial 3 finished with value: 0.6293706293706294 and parameters: {'model__C': 0.005454009910403115, 'model__gamma': 0.00041932691780907676, 

{'model__C': 0.054630769919335714,
 'model__gamma': 0.3684537529091605,
 'model__kernel': 'linear'}

- On définit les hyperparams directement dans trial.suggest_*.

In [15]:
import optuna

def objective(trial):

    C = trial.suggest_float("C", 1e-3, 1e3, log=True)
    gamma = trial.suggest_float("gamma", 1e-4, 1, log=True)
    kernel = trial.suggest_categorical("kernel", ["rbf", "linear"])

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(C=C, gamma=gamma, kernel=kernel))
    ])

    pipe.fit(X_train, y_train)
    return pipe.score(X_test, y_test)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

study.best_params


[I 2026-01-22 16:20:26,990] A new study created in memory with name: no-name-287526a2-95ce-4a92-8670-249e90b4f0c3
[I 2026-01-22 16:20:27,012] Trial 0 finished with value: 0.9370629370629371 and parameters: {'C': 52.62855657354243, 'gamma': 0.001204617706468866, 'kernel': 'linear'}. Best is trial 0 with value: 0.9370629370629371.
[I 2026-01-22 16:20:27,022] Trial 1 finished with value: 0.9440559440559441 and parameters: {'C': 3.458060959644137, 'gamma': 0.00016803563237014128, 'kernel': 'rbf'}. Best is trial 1 with value: 0.9440559440559441.
[I 2026-01-22 16:20:27,026] Trial 2 finished with value: 0.986013986013986 and parameters: {'C': 0.30538005158206116, 'gamma': 0.13033147030746164, 'kernel': 'linear'}. Best is trial 2 with value: 0.986013986013986.
[I 2026-01-22 16:20:27,032] Trial 3 finished with value: 0.986013986013986 and parameters: {'C': 0.2036383603185548, 'gamma': 0.00011515718568289265, 'kernel': 'linear'}. Best is trial 2 with value: 0.986013986013986.
[I 2026-01-22 16:20

{'C': 0.30538005158206116, 'gamma': 0.13033147030746164, 'kernel': 'linear'}

### Questions : go further

1️⃣ Tune a RandomForest :
      - n_estimators
      - max_depth
      - min_samples_split

2️⃣ Compare the performances Grid vs Random vs Optuna.

3️⃣ Test Optuna with 200 trees.

4️⃣ Visualize the curve of convergence Optuna.

1️⃣ Tune a RandomForest :
      - n_estimators
      - max_depth
      - min_samples_split

In [16]:
from sklearn.model_selection import cross_val_score

def rf_objective(trial):

    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 2, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42,
        n_jobs=-1
    )

    score = cross_val_score(
        model, X_train, y_train, cv=5, n_jobs=-1
    ).mean()

    return score


In [17]:
rf_study = optuna.create_study(direction="maximize")
rf_study.optimize(rf_objective, n_trials=30)

print("Best RF parameters:", rf_study.best_params)
print("Best RF CV score:", rf_study.best_value)


[I 2026-01-22 16:36:13,510] A new study created in memory with name: no-name-28dfa667-de14-41f6-a0d3-a9498ed8ff37
[I 2026-01-22 16:36:17,265] Trial 0 finished with value: 0.9600547195622436 and parameters: {'n_estimators': 270, 'max_depth': 24, 'min_samples_split': 6}. Best is trial 0 with value: 0.9600547195622436.
[I 2026-01-22 16:36:19,629] Trial 1 finished with value: 0.9577017783857729 and parameters: {'n_estimators': 208, 'max_depth': 20, 'min_samples_split': 3}. Best is trial 0 with value: 0.9600547195622436.
[I 2026-01-22 16:36:21,856] Trial 2 finished with value: 0.9483173734610124 and parameters: {'n_estimators': 139, 'max_depth': 3, 'min_samples_split': 9}. Best is trial 0 with value: 0.9600547195622436.
[I 2026-01-22 16:36:24,065] Trial 3 finished with value: 0.9483447332421342 and parameters: {'n_estimators': 76, 'max_depth': 23, 'min_samples_split': 10}. Best is trial 0 with value: 0.9600547195622436.
[I 2026-01-22 16:36:24,401] Trial 4 finished with value: 0.955348837209

Best RF parameters: {'n_estimators': 180, 'max_depth': 30, 'min_samples_split': 5}
Best RF CV score: 0.9624076607387142


2️⃣ Compare the performances Grid vs Random vs Optuna.


In [18]:
print("Baseline SVM accuracy :", pipe_svm.score(X_test, y_test))
print("GridSearch SVM accuracy:", grid.score(X_test, y_test))
print("RandomSearch SVM accuracy:", rand.score(X_test, y_test))
print("Optuna SVM accuracy:", study.best_value)


Baseline SVM accuracy : 0.9790209790209791
GridSearch SVM accuracy: 0.986013986013986
RandomSearch SVM accuracy: 0.986013986013986
Optuna SVM accuracy: 0.986013986013986


3️⃣ Test Optuna with 200 trees.

In [19]:
def rf_200_objective(trial):

    max_depth = trial.suggest_int("max_depth", 2, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42,
        n_jobs=-1
    )

    return cross_val_score(
        model, X_train, y_train, cv=5, n_jobs=-1
    ).mean()


In [20]:
rf_200_study = optuna.create_study(direction="maximize")
rf_200_study.optimize(rf_200_objective, n_trials=30)

print("Best parameters (200 trees):", rf_200_study.best_params)
print("Best CV score (200 trees):", rf_200_study.best_value)


[I 2026-01-22 16:38:03,923] A new study created in memory with name: no-name-4c74890e-c76d-4315-a18f-5ccd90924b27
[I 2026-01-22 16:38:04,299] Trial 0 finished with value: 0.9577017783857729 and parameters: {'max_depth': 26, 'min_samples_split': 5}. Best is trial 0 with value: 0.9577017783857729.
[I 2026-01-22 16:38:04,637] Trial 1 finished with value: 0.9600547195622436 and parameters: {'max_depth': 14, 'min_samples_split': 6}. Best is trial 1 with value: 0.9600547195622436.
[I 2026-01-22 16:38:05,048] Trial 2 finished with value: 0.9600547195622436 and parameters: {'max_depth': 20, 'min_samples_split': 6}. Best is trial 1 with value: 0.9600547195622436.
[I 2026-01-22 16:38:05,395] Trial 3 finished with value: 0.9506703146374831 and parameters: {'max_depth': 3, 'min_samples_split': 5}. Best is trial 1 with value: 0.9600547195622436.
[I 2026-01-22 16:38:05,719] Trial 4 finished with value: 0.9577017783857729 and parameters: {'max_depth': 18, 'min_samples_split': 5}. Best is trial 1 with

Best parameters (200 trees): {'max_depth': 14, 'min_samples_split': 6}
Best CV score (200 trees): 0.9600547195622436


4️⃣ Visualize the curve of convergence Optuna.

In [21]:
import optuna.visualization as vis

vis.plot_optimization_history(study)
